# 从零实现 RealNVP Normalizing Flow：可逆耦合层、精确似然与采样

Normalizing flow 用一串可逆映射把数据 $x$ 变到简单 base distribution $z$。与 VAE 不同，RealNVP 在维度不变且映射可逆时能用变量替换公式计算精确 log-likelihood。本 Notebook 手写 affine coupling、正逆传播、log-determinant、训练、采样与制品发布，不调用现成 flow 包。

受控二维四峰数据用于验证机制。似然下降和样本落在数据范围只是 smoke test，不能替代高维图像/密度估计中的 bits-per-dimension、OOD 反常似然、生成质量或校准评估。

In [ ]:
import copy
import hashlib
import io
import json
import math
import random
import warnings
from types import MappingProxyType

warnings.filterwarnings("ignore", message="The pynvml package is deprecated")
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F

SEED53 = 5301
random.seed(SEED53); np.random.seed(SEED53); torch.manual_seed(SEED53)
torch.set_num_threads(1)
DEVICE53 = torch.device("cpu")

def canonical_json53(value):
    return json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":"))

def sha53(value):
    return hashlib.sha256(value).hexdigest()

assert DEVICE53.type == "cpu" and torch.get_num_threads() == 1

## 1. 数据、切分与 train-only 标准化

四个 Gaussian 中心位于 `(-2,-2),(-2,2),(2,-2),(2,2)`，标准差 0.28。训练/验证/测试分别由独立 seed 生成，不能先合并再切分。normalizer 只由 train 推导：`x_norm=(x-mean_train)/std_train`。

张量合同是 `x:[B,2]`。二维数据便于画图，但这里不依赖可视化；数据 seed、中心、噪声、每个 split 和 normalizer 都进入发布 manifest。

In [ ]:
CENTERS53 = torch.tensor([[-2., -2.], [-2., 2.], [2., -2.], [2., 2.]])

def make_mixture53(count, seed, noise=0.28):
    if count < 4 or noise <= 0:
        raise ValueError("invalid_mixture_config")
    generator = torch.Generator().manual_seed(seed)
    labels = torch.randint(0, len(CENTERS53), (count,), generator=generator)
    points = CENTERS53[labels] + noise * torch.randn(count, 2, generator=generator)
    return points, labels

train_raw53, train_labels53 = make_mixture53(640, SEED53 + 1)
val_raw53, val_labels53 = make_mixture53(160, SEED53 + 2)
test_raw53, test_labels53 = make_mixture53(160, SEED53 + 3)
mean53 = train_raw53.mean(0); std53 = train_raw53.std(0, unbiased=False).clamp_min(1e-6)
normalize53 = lambda x: (x - mean53) / std53
train53, val53, test53 = map(normalize53, (train_raw53, val_raw53, test_raw53))
assert train53.shape == (640, 2) and val53.shape == test53.shape == (160, 2)
assert torch.allclose(train53.mean(0), torch.zeros(2), atol=1e-6)
assert torch.allclose(train53.std(0, unbiased=False), torch.ones(2), atol=1e-6)
regenerated53, _ = make_mixture53(640, SEED53 + 1)
assert torch.equal(regenerated53, train_raw53)
assert not torch.equal(train_raw53[:160], val_raw53)

## 2. Affine coupling 的三角 Jacobian

二值 mask `m:[D]` 把输入分为保留部分 $x_a=m\odot x$ 与变换部分：

$$y_a=x_a,\quad y_b=(1-m)\odot[x\odot\exp(s(x_a))+t(x_a)].$$

因为 $s,t$ 只读取保留坐标，Jacobian 为三角结构，$\log|\det J|=\sum_{b}s_b$。为防 `exp` 溢出，把网络输出经 `tanh` 限制到 `[-max_log_scale,max_log_scale]`。mask 必须同时含 0 和 1；全保留或全变换都不是有效 coupling。

In [ ]:
class CouplingNet53(nn.Module):
    def __init__(self, dim, hidden_dim):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(dim, hidden_dim), nn.Tanh(), nn.Linear(hidden_dim, hidden_dim), nn.Tanh(), nn.Linear(hidden_dim, 2 * dim))
        nn.init.zeros_(self.net[-1].weight); nn.init.zeros_(self.net[-1].bias)

    def forward(self, x):
        return self.net(x)

class AffineCoupling53(nn.Module):
    def __init__(self, mask, hidden_dim=32, max_log_scale=1.5):
        super().__init__()
        mask = torch.as_tensor(mask, dtype=torch.float32)
        if mask.ndim != 1 or not torch.all((mask == 0) | (mask == 1)) or mask.sum() in (0, mask.numel()):
            raise ValueError("mask_must_mix_zero_and_one")
        if not math.isfinite(max_log_scale) or not 0 < max_log_scale <= 5:
            raise ValueError("max_log_scale_finite_range_contract")
        self.register_buffer("mask", mask)
        self.max_log_scale = float(max_log_scale)
        self.conditioner = CouplingNet53(mask.numel(), hidden_dim)

    def _coupling_parameters(self, kept):
        raw_scale, shift = self.conditioner(kept).chunk(2, dim=-1)
        transformed = 1.0 - self.mask
        scale = self.max_log_scale * torch.tanh(raw_scale) * transformed
        shift = shift * transformed
        if not torch.isfinite(scale).all() or not torch.isfinite(shift).all():
            raise ValueError("nonfinite_coupling_parameters")
        return scale, shift

    def forward(self, x):
        if x.ndim != 2 or x.shape[1] != self.mask.numel() or not torch.isfinite(x).all():
            raise ValueError("coupling_input_contract")
        kept = x * self.mask; scale, shift = self._coupling_parameters(kept)
        y = kept + (1 - self.mask) * (x * torch.exp(scale) + shift)
        if not torch.isfinite(y).all():
            raise ValueError("nonfinite_coupling_output")
        return y, scale.sum(-1)

    def inverse(self, y):
        if y.ndim != 2 or y.shape[1] != self.mask.numel() or not torch.isfinite(y).all():
            raise ValueError("coupling_input_contract")
        kept = y * self.mask; scale, shift = self._coupling_parameters(kept)
        x = kept + (1 - self.mask) * ((y - shift) * torch.exp(-scale))
        if not torch.isfinite(x).all():
            raise ValueError("nonfinite_coupling_output")
        return x, -scale.sum(-1)

coupling_probe53 = AffineCoupling53([1, 0], hidden_dim=8)
x_probe53 = torch.tensor([[0.3, -0.7], [1.2, 0.4]])
y_probe53, ld_probe53 = coupling_probe53(x_probe53)
x_back53, ild_probe53 = coupling_probe53.inverse(y_probe53)
assert torch.allclose(x_probe53, x_back53, atol=1e-7)
assert torch.allclose(ld_probe53 + ild_probe53, torch.zeros(2), atol=1e-7)
try:
    AffineCoupling53([1, 1]); raise AssertionError("degenerate mask accepted")
except ValueError as error:
    assert str(error) == "mask_must_mix_zero_and_one"
try:
    AffineCoupling53([1,0],max_log_scale=float("inf")); raise AssertionError("infinite log scale accepted")
except ValueError as error:
    assert str(error) == "max_log_scale_finite_range_contract"

## 3. RealNVP 正向、逆向与精确 log-probability

多层 coupling 交替 `[1,0]` 与 `[0,1]`，使两个维度都能被变换。正向 `x -> z` 累加每层 log-det；逆向按层倒序执行。标准正态 base 下：

$$\log p_X(x)=\log p_Z(f(x))+\log|\det J_f(x)|.$$

`forward(x)` 返回 `z:[B,2], log_det:[B]`；`inverse(z)` 返回重建与逆 log-det；`sample(n,generator)` 从显式随机流采样后逆变换。

In [ ]:
class StandardNormal53:
    @staticmethod
    def log_prob(z):
        if z.ndim != 2 or not torch.isfinite(z).all():
            raise ValueError("base_input_contract")
        return -0.5 * (z.square() + math.log(2 * math.pi)).sum(-1)

    @staticmethod
    def sample(count, dim, generator):
        if count < 1 or dim < 1:
            raise ValueError("invalid_sample_shape")
        if not isinstance(generator, torch.Generator):
            raise TypeError("explicit_torch_generator_required")
        return torch.randn(count, dim, generator=generator)

class RealNVP53(nn.Module):
    def __init__(self, dim=2, hidden_dim=32, num_layers=6, max_log_scale=1.5):
        super().__init__()
        if dim != 2 or num_layers < 2:
            raise ValueError("this_teaching_model_requires_dim2_and_two_layers")
        masks = [[1, 0] if index % 2 == 0 else [0, 1] for index in range(num_layers)]
        self.dim, self.hidden_dim, self.num_layers, self.max_log_scale = dim, hidden_dim, num_layers, max_log_scale
        self.layers = nn.ModuleList([AffineCoupling53(mask, hidden_dim, max_log_scale) for mask in masks])

    def forward(self, x):
        z = x; total = torch.zeros(x.shape[0], dtype=x.dtype, device=x.device)
        for layer in self.layers:
            z, log_det = layer(z); total = total + log_det
        return z, total

    def inverse(self, z):
        x = z; total = torch.zeros(z.shape[0], dtype=z.dtype, device=z.device)
        for layer in reversed(self.layers):
            x, log_det = layer.inverse(x); total = total + log_det
        return x, total

    def log_prob(self, x):
        z, log_det = self(x)
        return StandardNormal53.log_prob(z) + log_det

    @torch.no_grad()
    def sample(self, count, generator):
        z = StandardNormal53.sample(count, self.dim, generator)
        return self.inverse(z)[0]

model_probe53 = RealNVP53(hidden_dim=12, num_layers=4)
z_probe53, total_ld53 = model_probe53(x_probe53)
reconstruction53, inverse_ld53 = model_probe53.inverse(z_probe53)
assert torch.allclose(reconstruction53, x_probe53, atol=1e-6)
assert torch.allclose(total_ld53 + inverse_ld53, torch.zeros(2), atol=1e-6)
assert torch.allclose(model_probe53.log_prob(x_probe53), StandardNormal53.log_prob(z_probe53) + total_ld53)
g1_53 = torch.Generator().manual_seed(3); g2_53 = torch.Generator().manual_seed(3)
assert torch.equal(model_probe53.sample(5, g1_53), model_probe53.sample(5, g2_53))
try:
    model_probe53.sample(2,None); raise AssertionError("implicit global RNG accepted")
except TypeError as error:
    assert str(error) == "explicit_torch_generator_required"

## 4. 用 autograd Jacobian 交叉验证 log-det

“正反能还原”仍可能同时把 log-det 写错。二维 oracle 对单点计算完整 Jacobian `J:[2,2]`，用 `slogdet(J)` 与模型累加值比较。这个测试会抓到漏乘 mask、逆向符号错误和 batch 维误求和。

初始化时最后一层为零，coupling 接近恒等映射；为了让 oracle 非平凡，给 conditioner 最后一层写入固定 bias。

In [ ]:
jacobian_layer53 = AffineCoupling53([1, 0], hidden_dim=6, max_log_scale=1.5).double()
with torch.no_grad():
    jacobian_layer53.conditioner.net[-1].bias.copy_(torch.tensor([0.0, 0.2, 0.0, -0.15], dtype=torch.float64))
point53 = torch.tensor([0.4, -0.2], dtype=torch.float64, requires_grad=True)
def transform_single53(value):
    return jacobian_layer53(value[None, :])[0].squeeze(0)
jacobian53 = torch.autograd.functional.jacobian(transform_single53, point53)
_, oracle_ld53 = jacobian_layer53(point53.detach()[None, :])
sign53, slogdet53 = torch.linalg.slogdet(jacobian53)
assert sign53 > 0 and torch.allclose(slogdet53, oracle_ld53.squeeze(0), atol=1e-10)
wrong_unscaled53 = jacobian_layer53.conditioner(point53.detach()[None, :] * jacobian_layer53.mask).chunk(2, -1)[0][0, 1]
assert not torch.allclose(slogdet53, wrong_unscaled53, atol=1e-5)
assert jacobian53.shape == (2, 2) and torch.isfinite(jacobian53).all()

## 5. 最大似然训练

训练最小化 `NLL=-mean(log_prob(x_train))`。每步用独立 generator 抽 minibatch；验证集只做选型，测试集最后一次报告。coupling 初始为恒等，初始 NLL 就是标准正态对标准化数据的拟合，后续下降说明 flow 学到了非 Gaussian 四峰结构。

这里只比较同一预处理、同一维度下的 NLL；不同 dequantization、normalizer 或数据支持集的似然不能直接横比。

In [ ]:
torch.manual_seed(SEED53)
model53 = RealNVP53(hidden_dim=32, num_layers=6).to(DEVICE53)
optimizer53 = torch.optim.Adam(model53.parameters(), lr=0.004)
batch_generator53 = torch.Generator().manual_seed(SEED53 + 10)
with torch.no_grad(): initial_nll53 = float(-model53.log_prob(val53).mean())
history53 = []
for step53 in range(320):
    index53 = torch.randint(0, train53.shape[0], (128,), generator=batch_generator53)
    loss53 = -model53.log_prob(train53[index53]).mean()
    optimizer53.zero_grad(set_to_none=True); loss53.backward()
    torch.nn.utils.clip_grad_norm_(model53.parameters(), 5.0); optimizer53.step()
    if step53 % 40 == 0: history53.append(float(loss53.detach()))
with torch.no_grad():
    final_val_nll53 = float(-model53.log_prob(val53).mean())
    test_nll53 = float(-model53.log_prob(test53).mean())
assert final_val_nll53 < initial_nll53 - 0.15
assert math.isfinite(test_nll53) and all(math.isfinite(value) for value in history53)
assert any(parameter.grad is not None and torch.isfinite(parameter.grad).all() for parameter in model53.parameters())
print({"initial_val_nll": round(initial_nll53, 4), "final_val_nll": round(final_val_nll53, 4), "test_nll": round(test_nll53, 4)})

## 6. 采样、支持集与 OOD 边界

采样路径是 `z~N(0,I) -> inverse(z) -> denormalize`。必须复用一个显式 generator，不能在循环或每个 batch 内重置 seed。由于可逆 flow 通常对整个 $\mathbb R^D$ 给正密度，“高似然”不自动等于“属于训练语义”；高维 flow 尤其可能给 OOD 更高似然。

这里检查样本有限、正逆一致以及多数样本靠近某个中心，只作为受控分布的 sanity check。

In [ ]:
sample_generator53 = torch.Generator().manual_seed(5319)
samples_norm53 = model53.sample(400, sample_generator53)
samples_raw53 = samples_norm53 * std53 + mean53
distances53 = torch.cdist(samples_raw53, CENTERS53).min(-1).values
assert samples_raw53.shape == (400, 2) and torch.isfinite(samples_raw53).all()
assert float((distances53 < 1.2).float().mean()) > 0.65
z_samples53, ld_samples53 = model53(samples_norm53[:20])
recovered_samples53, ild_samples53 = model53.inverse(z_samples53)
assert torch.allclose(recovered_samples53, samples_norm53[:20], atol=2e-5)
assert torch.allclose(ld_samples53 + ild_samples53, torch.zeros(20), atol=2e-5)
try:
    model53(torch.tensor([[float("inf"), 0.0]])); raise AssertionError("infinite sample accepted")
except ValueError as error:
    assert str(error) == "coupling_input_contract"

## 7. 标准化也会改变密度单位

模型学习的是标准化坐标 $u=(x-\mu)/\sigma$ 上的密度。若服务要返回原始坐标密度，还必须再应用一次变量替换：

$$\log p_X(x)=\log p_U((x-\mu)/\sigma)-\sum_d\log\sigma_d.$$

漏掉这一项不会影响同一 normalizer 下的排序，却会让 NLL、阈值和跨版本比较整体偏移。因此 mean/std 不只是前处理参数，也是密度语义的一部分。

In [ ]:
def raw_log_prob53(model, raw_points, mean, std):
    if raw_points.ndim != 2 or mean.shape != std.shape or mean.shape != (raw_points.shape[1],):
        raise ValueError("raw_density_shape_contract")
    if raw_points.dtype != mean.dtype or raw_points.dtype != std.dtype or raw_points.device != mean.device or raw_points.device != std.device:
        raise ValueError("raw_density_dtype_device_contract")
    if not torch.isfinite(raw_points).all() or not torch.isfinite(mean).all() or not torch.isfinite(std).all() or not torch.all(std > 0):
        raise ValueError("raw_density_finite_positive_contract")
    normalized = (raw_points - mean) / std
    return model.log_prob(normalized) - std.log().sum()

raw_lp53 = raw_log_prob53(model53, test_raw53[:6], mean53, std53)
manual_raw_lp53 = model53.log_prob(test53[:6]) - std53.log().sum()
assert torch.allclose(raw_lp53, manual_raw_lp53, atol=1e-7)
shifted_mean53 = mean53 + 0.5
assert not torch.allclose(raw_log_prob53(model53, test_raw53[:6], shifted_mean53, std53), raw_lp53)
try:
    raw_log_prob53(model53, test_raw53[:2], mean53, torch.tensor([1.0, 0.0])); raise AssertionError("zero scale accepted")
except ValueError as error:
    assert str(error) == "raw_density_finite_positive_contract"
try:
    raw_log_prob53(model53,test_raw53[:2],mean53,torch.tensor([float("inf"),1.])); raise AssertionError("infinite std accepted")
except ValueError as error:
    assert str(error) == "raw_density_finite_positive_contract"

## 8. 发布合同与整体重签攻击

manifest 绑定：模型层数/hidden/scale clamp、交替 mask、标准正态 base、四个中心与噪声、split seed/count/hash、train-only mean/std、训练步数与随机流。loader 重新生成三个 split 并重算 normalizer，而不是只比较调用方提供的统计量。

publisher registry 位于 package 外并只读；state 语义摘要包含每个 tensor 的 key、dtype、shape、bytes。loader 返回 `PublishedFlow53`，把 train-only mean/std、原单位 `raw_log_prob` 与 `sample_raw` 封装在一起，避免调用方忘记密度单位的 Jacobian。替换模型后重算内部所有摘要仍会得到新 bundle，因此被 registry 拒绝。

In [ ]:
def tensor_hash53(tensor):
    value = tensor.detach().cpu().contiguous()
    payload = str(value.dtype).encode() + canonical_json53(list(value.shape)).encode() + value.numpy().tobytes()
    return sha53(payload)

def dataset_hash53(points, labels):
    return sha53((tensor_hash53(points) + tensor_hash53(labels)).encode())

def state_digest53(state):
    digest = hashlib.sha256()
    for key, tensor in sorted(state.items()):
        digest.update(key.encode()); digest.update(tensor_hash53(tensor).encode())
    return digest.hexdigest()

manifest53 = {
    "artifact_id": "realnvp-four-mode-v1", "version": 1,
    "model_config": {"dim": 2, "hidden_dim": 32, "num_layers": 6, "max_log_scale": 1.5},
    "base": "standard_normal_2d", "masks": [[1, 0], [0, 1]] * 3,
    "data": {"centers": CENTERS53.tolist(), "noise": 0.28,
             "splits": {"train": [640, SEED53 + 1, dataset_hash53(train_raw53, train_labels53)],
                        "val": [160, SEED53 + 2, dataset_hash53(val_raw53, val_labels53)],
                        "test": [160, SEED53 + 3, dataset_hash53(test_raw53, test_labels53)]}},
    "preprocess": {"kind": "train_only_standardize", "mean": mean53.tolist(), "std": std53.tolist()},
    "training": {"seed": SEED53, "optimizer": "Adam", "steps": 320, "batch_size": 128, "lr": 0.004, "grad_clip_norm": 5.0},
}

def build_package53(model, manifest):
    buffer = io.BytesIO(); torch.save(model.state_dict(), buffer); raw = buffer.getvalue()
    state = torch.load(io.BytesIO(raw), map_location="cpu", weights_only=True)
    manifest_sha = sha53(canonical_json53(manifest).encode()); semantic = state_digest53(state); raw_sha = sha53(raw)
    bundle = sha53(canonical_json53({"manifest_sha": manifest_sha, "state_digest": semantic, "state_bytes_sha": raw_sha}).encode())
    return {"manifest": copy.deepcopy(manifest), "manifest_sha": manifest_sha, "state_bytes": raw,
            "state_digest": semantic, "state_bytes_sha": raw_sha, "bundle_digest": bundle}

package53 = build_package53(model53, manifest53)
PUBLISHER_REGISTRY53 = MappingProxyType({("realnvp-four-mode-v1", 1): package53["bundle_digest"]})

class PublishedFlow53:
    def __init__(self, model, mean, std):
        self._model = model
        self._mean = mean.detach().clone()
        self._std = std.detach().clone()
    @property
    def mean(self): return self._mean.clone()
    @property
    def std(self): return self._std.clone()
    def raw_log_prob(self, raw_points): return raw_log_prob53(self._model,raw_points,self._mean,self._std)
    @torch.no_grad()
    def sample_raw(self,count,generator): return self._model.sample(count,generator)*self._std+self._mean

def load_flow53(package):
    manifest = package["manifest"]; key = (manifest.get("artifact_id"), manifest.get("version"))
    if PUBLISHER_REGISTRY53.get(key) != package.get("bundle_digest"):
        raise RuntimeError("publisher_registry_rejected_bundle")
    if manifest != manifest53 or sha53(canonical_json53(manifest).encode()) != package["manifest_sha"]:
        raise RuntimeError("manifest_contract_mismatch")
    regenerated = [make_mixture53(count, seed) for count, seed, _ in manifest["data"]["splits"].values()]
    for (points, labels), (_, _, expected) in zip(regenerated, manifest["data"]["splits"].values()):
        if dataset_hash53(points, labels) != expected: raise RuntimeError("split_snapshot_mismatch")
    train_points = regenerated[0][0]; derived_mean = train_points.mean(0); derived_std = train_points.std(0, unbiased=False).clamp_min(1e-6)
    if not torch.allclose(derived_mean, torch.tensor(manifest["preprocess"]["mean"])) or not torch.allclose(derived_std, torch.tensor(manifest["preprocess"]["std"])):
        raise RuntimeError("preprocess_not_derived_from_train")
    if sha53(package["state_bytes"]) != package["state_bytes_sha"]: raise RuntimeError("state_bytes_mismatch")
    state = torch.load(io.BytesIO(package["state_bytes"]), map_location="cpu", weights_only=True)
    if state_digest53(state) != package["state_digest"]: raise RuntimeError("state_semantic_mismatch")
    expected_bundle = sha53(canonical_json53({"manifest_sha": package["manifest_sha"], "state_digest": package["state_digest"], "state_bytes_sha": package["state_bytes_sha"]}).encode())
    if expected_bundle != package["bundle_digest"]: raise RuntimeError("bundle_digest_mismatch")
    restored = RealNVP53(**manifest["model_config"]); restored.load_state_dict(state); restored.eval()
    return PublishedFlow53(restored,derived_mean,derived_std)

restored53 = load_flow53(package53)
assert torch.allclose(restored53.raw_log_prob(test_raw53[:8]), raw_log_prob53(model53,test_raw53[:8],mean53,std53), atol=1e-7)
published_samples53 = restored53.sample_raw(4,torch.Generator().manual_seed(88))
direct_samples53 = model53.sample(4,torch.Generator().manual_seed(88))*std53+mean53
assert torch.equal(published_samples53,direct_samples53)
leaked_mean53 = restored53.mean; leaked_mean53.add_(100)
assert torch.allclose(restored53.mean,mean53)
forged53 = RealNVP53(hidden_dim=32, num_layers=6)
forged_package53 = build_package53(forged53, manifest53)
try:
    load_flow53(forged_package53); raise AssertionError("re-signed forged flow accepted")
except RuntimeError as error:
    assert str(error) == "publisher_registry_rejected_bundle"
assert isinstance(PUBLISHER_REGISTRY53, MappingProxyType)

## 9. 复杂度、失败模式与原始来源

每层 conditioner 的成本约为 MLP 成本，训练必须同时跑全部 coupling；采样走逆序但无需迭代数百个 diffusion step。常见错误是漏加 log-det、逆变换符号错误、mask 不交替、scale 无界、用全数据标准化、在离散像素上忘记 dequantization，以及用 likelihood 单独判断 OOD。

- Dinh, Sohl-Dickstein & Bengio, [Density Estimation using Real NVP](https://arxiv.org/abs/1605.08803), ICLR 2017。
- Rezende & Mohamed, [Variational Inference with Normalizing Flows](https://arxiv.org/abs/1505.05770), ICML 2015。
- Papamakarios et al., [Normalizing Flows for Probabilistic Modeling and Inference](https://arxiv.org/abs/1912.02762), JMLR 2021。

本例只复现二维 affine coupling 与精确变量替换，没有覆盖图像多尺度 squeeze、invertible 1x1 convolution 或高维生产训练。